In [1]:
import numpy as np
import matplotlib.pyplot as plt


In [23]:
def Plot_Solution_Transient(u, v, p, Nx, Ny, Lx, Ly): 

    X, Y = GetMeshGrid(Nx, Ny, Lx, Ly)
    plt.figure(figsize=[11 ,11])
    cp = plt.contourf(X, Y, p, levels = 20)
    plt.quiver(X, Y, u, v)
    plt.colorbar(cp)

    plt.show()
    return

def CalculateCentralDifference(F, dx, dy):
    dFdx = (F[2:, 1:-1] - F[:-2, 1:-1])/(2*dx)
    dFdy = (F[1:-1, 2:] - F[1:-1, :-2])/(2*dy)
    return dFdx, dFdy

def CalculateCentralDifference2nd(F, dx, dy):
    ddFdx = (F[2:, 1:-1] - 2 * F[1:-1, 1:-1] + F[:-2, 1:-1])/(2*dx)
    ddFdy = (F[1:-1, 2:] - 2 * F[1:-1, 1:-1] + F[1:-1, :-2])/(2*dy)
    return ddFdx, ddFdy

def SIMPLE_Solver(Nx, Ny, dx, dy, dt, L, numTimesteps):
    u = np.zeros((Nx * Ny, 1))
    v = np.zeros((Nx * Ny, 1))
    p = np.zeros((Nx * Ny, 1))

    for n in range(numTimesteps):
        u, v = ImplicitSolveNavierStokes(u, v, p, Nx, Ny, dx, dy, dt, L)

    return u, v, p

def ImplicitSolveNavierStokes(u, v, p, Nx, Ny, dx, dy, dt, L):
    u = u.reshape(-1, 1)
    v = v.reshape(-1, 1)

    #Get pressure and set up RHS
    #dpdx, dpdy = CalculateCentralDifference(p, dx, dy)
    dpdx, dpdy = 0

    bx = -dpdx * np.ones((Nx * Ny, 1)) + u
    by = -dpdy * np.ones((Nx * Ny, 1)) + v

    #Gonna have to do some picard iteration
    for i in range(30):
        #Have to build new matrix each iteration
        A = BuildMatrix(u, v, Nx, Ny, dx, dy, dy, L)

        #Should probably replace with a better solver here
        um = np.linalg.inv(A) @ bx
        vm = np.linalg.inv(A) @ by

        u = um
        v = vm

    u = u.reshape(Ny, Nx)
    v = v.reshape(Ny, Nx)

    return u, v

def BuildMatrix(u, v, Nx, Ny, dx, dy, dt, L):
    size = Nx * Ny - 2
    
    betax = (dt * L)/ dx**2
    betay = (dt * L)/ dy**2
    
    Cx = dt/(2 * dx)
    Cy = dt/(2 * dy)
    print(betax, betay, Cx, Cy)
    #i,j
    A = (1 + 2*betax + 2*betay) * np.identity(size) 
    #i+1, j
    A += (Cx * u[1:-1] - betax) * np.eye(size, k = 1) 
    #i-1, j
    A += (-Cx * u[1:-1] - betax) * np.eye(size, k = -1) 
    #i, j+1
    A += (Cy * v[1:-1] - betay) * np.eye(size, k = Nx) 
    #i, j-1
    A += (-Cx * v[1:-1] - betay) * np.eye(size, k = -Nx)

    return A

In [24]:
#  Input Data #
Lx = 1
Ly = 1
dx = 0.01
dy = 0.01
dt = 0.005

pho = 1000
visc = pho * 8.4e-4
tmax = 2

numTimesteps = int(tmax/dt)

Nx = int(Lx/dx) + 1
Ny = int(Ly/dy) + 1

u = np.zeros((Nx, Ny))
v = np.zeros((Nx, Ny))

u = u.reshape(-1, 1)
v = v.reshape(-1, 1)

betax = BuildMatrix(u, v, Nx, Ny, dx, dy, dt, visc)
print(betax)
#print(betax, betay, Cx, Cy)


42.00000000000001 42.00000000000001 0.25 0.25
[[169. -42.   0. ...   0.   0.   0.]
 [-42. 169. -42. ...   0.   0.   0.]
 [  0. -42. 169. ...   0.   0.   0.]
 ...
 [  0.   0.   0. ... 169. -42.   0.]
 [  0.   0.   0. ... -42. 169. -42.]
 [  0.   0.   0. ...   0. -42. 169.]]
